# Etapa 1 — Auditoria dos dados

**Pergunta central:** Que base temos de fato, quais problemas ou particularidades ela apresenta e quais decisões de modelagem poderão ser tomadas a partir disso?

**Comentário Técnico:** Esta etapa é exclusivamente descritiva e diagnóstica. Não há imputação, substituição de valores especiais, seleção de variáveis, definição de amostras temporais ou modelagem. `Ever30Mob6` é tratado somente como indicador de evento adverso.

In [1]:
from pathlib import Path
import hashlib
import sys
import warnings

import numpy as np
import openpyxl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src' / 'behavior_score').exists():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src' / 'behavior_score').exists():
    raise RuntimeError('Execute o notebook a partir da raiz do projeto ou da pasta notebooks/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.behavior_score.config import (
    ALVO, COLUNA_ID, COLUNA_INDICE, COLUNA_SAFRA, PASTA_DADOS_BRUTOS,
    PASTA_FIGURAS, PASTA_TABELAS, VARIAVEIS_CATEGORICAS,
    VARIAVEIS_MODELO, VARIAVEIS_NUMERICAS,
)
from src.behavior_score.visualization import (
    CORES, aplicar_eixo_percentual, aplicar_layout_executivo, salvar_grafico,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
arquivos_excel = sorted(PASTA_DADOS_BRUTOS.glob('*.xlsx'))
if len(arquivos_excel) != 1:
    raise RuntimeError(f'Esperado exatamente 1 arquivo .xlsx em data/raw; encontrados: {len(arquivos_excel)}')
caminho_base = arquivos_excel[0]
hash_base = hashlib.sha256(caminho_base.read_bytes()).hexdigest()
arquivo_excel = pd.ExcelFile(caminho_base, engine='openpyxl')
if len(arquivo_excel.sheet_names) != 1:
    raise RuntimeError(f'Esperada exatamente uma aba; encontradas: {arquivo_excel.sheet_names}')
nome_aba = arquivo_excel.sheet_names[0]
base = pd.read_excel(caminho_base, sheet_name=nome_aba, index_col=None, engine='openpyxl')
workbook_validacao = openpyxl.load_workbook(caminho_base, read_only=True, data_only=True)
worksheet_validacao = workbook_validacao[nome_aba]
cabecalho_openpyxl = [celula.value for celula in next(worksheet_validacao.iter_rows(min_row=1, max_row=1))]
dimensao_excel = (worksheet_validacao.max_row, worksheet_validacao.max_column)
workbook_validacao.close()
assert cabecalho_openpyxl == base.columns.tolist(), 'Cabeçalhos de pandas e openpyxl divergem.'
assert base.index.equals(pd.RangeIndex(len(base))) and base.index.name is None
base_original = base.copy(deep=True)
print(f'Base localizada: {caminho_base.relative_to(RAIZ)}')
print(f'Aba: {nome_aba} | SHA-256: {hash_base}')
print(f'Excel físico: {dimensao_excel[0]:,} linhas com cabeçalho × {dimensao_excel[1]} colunas')
print(f'DataFrame: {base.shape[0]:,} registros × {base.shape[1]} colunas | índice: RangeIndex não nomeado')

Base localizada: data\raw\base_behavior_score.xlsx
Aba: case | SHA-256: 7ae6cca5ca1e488920a465c1f6fe93850c972f15a996eb1bc8f3a35783435e08
Excel físico: 200,044 linhas com cabeçalho × 18 colunas
DataFrame: 200,043 registros × 18 colunas | índice: RangeIndex não nomeado


## 1. Que estrutura foi efetivamente recebida?

A comparação abaixo usa `config.py` como contrato esperado e não corrige inconsistências automaticamente.

In [2]:
colunas_esperadas = [COLUNA_INDICE, COLUNA_ID, COLUNA_SAFRA, ALVO, *VARIAVEIS_MODELO]
colunas_encontradas = base.columns.tolist()
colunas_ausentes = [c for c in colunas_esperadas if c not in colunas_encontradas]
colunas_adicionais = [c for c in colunas_encontradas if c not in colunas_esperadas]
tabela_fonte = pd.DataFrame({
    'arquivo': [str(caminho_base.relative_to(RAIZ))], 'aba': [nome_aba], 'sha256': [hash_base],
    'linhas_excel_com_cabecalho': [dimensao_excel[0]], 'colunas_excel': [dimensao_excel[1]],
    'registros_dataframe': [len(base)], 'colunas_dataframe': [base.shape[1]],
    'index_col_usado': [False], 'tipo_indice_dataframe': [type(base.index).__name__],
})
tabela_estrutura = pd.DataFrame({
    'coluna': sorted(set(colunas_esperadas) | set(colunas_encontradas)),
}).assign(
    esperada=lambda d: d['coluna'].isin(colunas_esperadas),
    encontrada=lambda d: d['coluna'].isin(colunas_encontradas),
)
tabela_estrutura['tipo_encontrado'] = tabela_estrutura['coluna'].map(base.dtypes.astype(str))
display(tabela_fonte)
display(pd.DataFrame({
    'metrica': ['linhas', 'colunas', 'memoria_mb', 'colunas_ausentes', 'colunas_adicionais'],
    'valor': [len(base), base.shape[1], round(base.memory_usage(deep=True).sum() / 1024**2, 2),
              ', '.join(colunas_ausentes) or 'nenhuma', ', '.join(colunas_adicionais) or 'nenhuma'],
}))
display(tabela_estrutura)
display(base.head())
display(Markdown(
    f"**Análise/Interpretação:** Foram encontradas **{base.shape[1]} colunas** e "
    f"**{len(base):,} observações**. Colunas ausentes frente ao contrato: "
    f"**{', '.join(colunas_ausentes) or 'nenhuma'}**. Colunas adicionais: "
    f"**{', '.join(colunas_adicionais) or 'nenhuma'}**. Uma divergência de contrato deve ser "
    "avaliada antes de preparar atributos; não foi feita correção automática. A ausência de `index` foi "
    "confirmada também no cabeçalho físico via openpyxl; o RangeIndex é apenas o índice padrão criado pelo pandas."
))

,arquivo,aba,sha256,linhas_excel_com_cabecalho,colunas_excel,registros_dataframe,colunas_dataframe,index_col_usado,tipo_indice_dataframe
0,data\raw\base_behavior_score.xlsx,case,7ae6cca5ca1e488920a465c1f6fe93850c972f15a996eb...,200044,18,200043,18,False,RangeIndex


,metrica,valor
0,linhas,200043
1,colunas,18
2,memoria_mb,27.47
3,colunas_ausentes,index
4,colunas_adicionais,nenhuma


,coluna,esperada,encontrada,tipo_encontrado
0,Ever30Mob6,True,True,int64
1,cat_var10,True,True,int64
2,cat_var13,True,True,float64
3,cat_var15,True,True,float64
4,cat_var2,True,True,float64
5,cat_var6,True,True,int64
6,data_ref_safra,True,True,int64
7,id,True,True,int64
8,index,True,False,NaN
9,var1,True,True,float64


,data_ref_safra,id,Ever30Mob6,var1,cat_var2,var3,var4,var5,cat_var6,var7,var8,var9,cat_var10,var11,var12,cat_var13,var14,cat_var15
0,201908,1,0,98.00,1.000000,97.49,1.000000,1.000000,12,0.00,1.000000,1.000000,0,1.000000,99999.0,3.0,1.000000,6.0
1,202001,2,0,289.51,1.000000,289.51,1.000000,1.000000,12,1346.35,1.000000,1.000000,0,1.000000,99999.0,NaN,1.000000,8.0
2,201908,3,0,106.16,1.000000,106.16,1.000000,0.833333,5,717.91,1.000000,0.833333,7,1.000000,99998.0,3.0,0.833333,0.0
3,201912,4,0,1243.32,0.666667,1243.32,0.870769,0.583333,12,50.87,0.851072,0.500000,0,0.821816,99997.0,NaN,0.583333,12.0
4,201905,5,0,184.36,1.000000,184.36,1.000000,1.000000,11,883.10,1.000000,1.000000,0,1.000000,99999.0,NaN,0.916667,0.0


**Análise/Interpretação:** Foram encontradas **18 colunas** e **200,043 observações**. Colunas ausentes frente ao contrato: **index**. Colunas adicionais: **nenhuma**. Uma divergência de contrato deve ser avaliada antes de preparar atributos; não foi feita correção automática. A ausência de `index` foi confirmada também no cabeçalho físico via openpyxl; o RangeIndex é apenas o índice padrão criado pelo pandas.

## 2. Qual é a granularidade observada e existem duplicidades?

**Comentário Técnico:** `id` não é presumido único. A relação com o índice físico e com as safras é examinada empiricamente.

In [3]:
ids_unicos = base[COLUNA_ID].nunique(dropna=True)
ids_repetidos = base.loc[base[COLUNA_ID].duplicated(keep=False), COLUNA_ID].nunique(dropna=True)
linhas_duplicadas = int(base.duplicated().sum())
ids_multiplas_safras = int((base.groupby(COLUNA_ID, dropna=False)[COLUNA_SAFRA].nunique() > 1).sum())
indice_explicito_presente = COLUNA_INDICE in base.columns
indice_padrao_sequencial = base.index.equals(pd.RangeIndex(len(base)))
relacao_indice_id = 'não aplicável: coluna index ausente'
if indice_explicito_presente:
    relacao_indice_id = str(base[COLUNA_INDICE].equals(base[COLUNA_ID]))
tabela_identificadores = pd.DataFrame({
    'metrica': ['registros', 'ids_unicos', 'ids_repetidos', 'ids_em_multiplas_safras',
                'linhas_totalmente_duplicadas', 'coluna_index_presente',
                'indice_dataframe_sequencial', 'coluna_index_igual_id'],
    'valor': [len(base), ids_unicos, ids_repetidos, ids_multiplas_safras, linhas_duplicadas,
              indice_explicito_presente, indice_padrao_sequencial, relacao_indice_id],
})
display(tabela_identificadores)
display(Markdown(
    f"**Análise/Interpretação:** Há **{ids_unicos:,} IDs únicos**, **{ids_repetidos:,} IDs repetidos** "
    f"e **{ids_multiplas_safras:,} IDs em múltiplas safras**. Foram encontradas "
    f"**{linhas_duplicadas:,} linhas totalmente duplicadas**. Esses resultados determinam se a unidade "
    "observada se comporta como cliente, cliente-safra ou outra granularidade e precisam ser preservados "
    "no futuro desenho amostral."
))

,metrica,valor
0,registros,200043
1,ids_unicos,200043
2,ids_repetidos,0
3,ids_em_multiplas_safras,0
4,linhas_totalmente_duplicadas,0
5,coluna_index_presente,False
6,indice_dataframe_sequencial,True
7,coluna_index_igual_id,não aplicável: coluna index ausente


**Análise/Interpretação:** Há **200,043 IDs únicos**, **0 IDs repetidos** e **0 IDs em múltiplas safras**. Foram encontradas **0 linhas totalmente duplicadas**. Esses resultados determinam se a unidade observada se comporta como cliente, cliente-safra ou outra granularidade e precisam ser preservados no futuro desenho amostral.

## 3. Quais safras existem e como a população evolui?

Nenhuma divisão Treino/Validação/OOT é definida nesta auditoria.

In [4]:
safra_texto = base[COLUNA_SAFRA].astype('Int64').astype('string')
safra_data = pd.to_datetime(safra_texto, format='%Y%m', errors='coerce')
if safra_data.isna().any():
    raise ValueError(f'Existem {safra_data.isna().sum()} safras inválidas no formato AAAAMM.')
tabela_safras = (
    pd.DataFrame({'safra': safra_data})
    .value_counts('safra').rename('quantidade_registros').reset_index().sort_values('safra')
)
tabela_safras['percentual_populacao'] = tabela_safras['quantidade_registros'] / len(base)
sequencia_completa = pd.date_range(tabela_safras['safra'].min(), tabela_safras['safra'].max(), freq='MS')
lacunas_safras = sequencia_completa.difference(pd.DatetimeIndex(tabela_safras['safra']))
base_ordenada_por_safra = bool(safra_data.is_monotonic_increasing)
display(tabela_safras.style.format({'safra': lambda x: x.strftime('%Y-%m'), 'percentual_populacao': '{:.2%}'}))
fig_populacao = go.Figure(go.Bar(
    x=tabela_safras['safra'], y=tabela_safras['quantidade_registros'], marker_color=CORES['principal'],
    hovertemplate='Safra: %{x|%Y-%m}<br>Registros: %{y:,}<extra></extra>',
))
aplicar_layout_executivo(fig_populacao, 'População por safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Quantidade de registros', mostrar_legenda=False)
fig_populacao.show()
display(Markdown(
    f"**Análise/Interpretação:** A base contém **{len(tabela_safras)} safras**, de "
    f"**{tabela_safras['safra'].min():%Y-%m}** a **{tabela_safras['safra'].max():%Y-%m}**. "
    f"A expectativa documental é 13. Lacunas mensais: **{', '.join(d.strftime('%Y-%m') for d in lacunas_safras) or 'nenhuma'}**. "
    f"A ordem original por safra é **{'cronológica' if base_ordenada_por_safra else 'não cronológica (embaralhada)'}**. "
    "A escolha das janelas temporais permanece pendente até revisão conjunta destes resultados."
))

,safra,quantidade_registros,percentual_populacao
9,2019-01,13936,6.97%
10,2019-02,13838,6.92%
12,2019-03,13783,6.89%
11,2019-04,13825,6.91%
8,2019-05,14182,7.09%
7,2019-06,14503,7.25%
6,2019-07,14808,7.40%
5,2019-08,15449,7.72%
4,2019-09,16134,8.07%
3,2019-10,16487,8.24%


**Análise/Interpretação:** A base contém **13 safras**, de **2019-01** a **2020-01**. A expectativa documental é 13. Lacunas mensais: **nenhuma**. A ordem original por safra é **não cronológica (embaralhada)**. A escolha das janelas temporais permanece pendente até revisão conjunta destes resultados.

## 4. O alvo aparenta ser binário e é estável por safra?

In [5]:
valores_alvo = sorted(base[ALVO].dropna().unique().tolist())
alvo_binario = set(valores_alvo).issubset({0, 1}) and len(valores_alvo) == 2
tabela_alvo = (base[ALVO].value_counts(dropna=False).rename_axis('valor').rename('quantidade').reset_index())
tabela_alvo['percentual'] = tabela_alvo['quantidade'] / len(base)
tabela_evento_safra = (
    base.assign(safra=safra_data).groupby('safra', as_index=False)[ALVO]
    .agg(quantidade_registros='size', quantidade_eventos='sum', taxa_evento='mean')
)
display(tabela_alvo.style.format({'percentual': '{:.2%}'}))
display(tabela_evento_safra.style.format({'safra': lambda x: x.strftime('%Y-%m'), 'taxa_evento': '{:.2%}'}))
fig_evento = go.Figure(go.Scatter(
    x=tabela_evento_safra['safra'], y=tabela_evento_safra['taxa_evento'], mode='lines+markers',
    line=dict(color=CORES['principal'], width=3), marker=dict(size=8),
    hovertemplate='Safra: %{x|%Y-%m}<br>Taxa do evento: %{y:.2%}<extra></extra>',
))
aplicar_layout_executivo(fig_evento, 'Taxa do evento por safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Taxa do evento', mostrar_legenda=False)
aplicar_eixo_percentual(fig_evento)
fig_evento.show()
salvar_grafico(fig_evento, '00_01_taxa_evento_por_safra', PASTA_FIGURAS)
display(Markdown(
    f"**Análise/Interpretação:** Valores não ausentes observados no alvo: **{valores_alvo}**; "
    f"alvo binário completo: **{alvo_binario}**; ausentes: **{base[ALVO].isna().sum():,}**. "
    f"A taxa global do evento adverso é **{base[ALVO].mean():.2%}** e varia de "
    f"**{tabela_evento_safra['taxa_evento'].min():.2%}** a **{tabela_evento_safra['taxa_evento'].max():.2%}** entre safras. "
    "Essa variação deve orientar a futura validação temporal, sem definir os cortes nesta etapa."
))

,valor,quantidade,percentual
0,0,174825,87.39%
1,1,25218,12.61%


,safra,quantidade_registros,quantidade_eventos,taxa_evento
0,2019-01,13936,1512,10.85%
1,2019-02,13838,1601,11.57%
2,2019-03,13783,1598,11.59%
3,2019-04,13825,1675,12.12%
4,2019-05,14182,1662,11.72%
5,2019-06,14503,1734,11.96%
6,2019-07,14808,1737,11.73%
7,2019-08,15449,1819,11.77%
8,2019-09,16134,1993,12.35%
9,2019-10,16487,2100,12.74%


C:\GitHub\datascience\projetos\credit-card-behavior-score\src\behavior_score\visualization.py:187: RuntimeWarning: Não foi possível exportar o gráfico para PNG. O arquivo HTML foi preservado. Detalhe: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

  warnings.warn(


**Análise/Interpretação:** Valores não ausentes observados no alvo: **[0, 1]**; alvo binário completo: **True**; ausentes: **0**. A taxa global do evento adverso é **12.61%** e varia de **10.85%** a **17.47%** entre safras. Essa variação deve orientar a futura validação temporal, sem definir os cortes nesta etapa.

## 5. Onde existem valores ausentes e há mudança temporal?

In [6]:
tabela_missing = pd.DataFrame({
    'variavel': base.columns,
    'quantidade_ausentes': base.isna().sum().values,
    'percentual_ausentes': base.isna().mean().values,
    'quantidade_valores_unicos': base.nunique(dropna=True).values,
}).sort_values(['percentual_ausentes', 'variavel'], ascending=[False, True])
variaveis_com_missing = tabela_missing.loc[tabela_missing['quantidade_ausentes'] > 0, 'variavel'].tolist()
display(tabela_missing.style.format({'percentual_ausentes': '{:.2%}'}))
missing_por_safra = pd.DataFrame()
if variaveis_com_missing:
    missing_por_safra = (base.assign(safra=safra_data).groupby('safra')[variaveis_com_missing]
                          .agg(lambda s: s.isna().mean()).T)
    fig_missing = px.imshow(
        missing_por_safra, aspect='auto', color_continuous_scale=[CORES['fundo'], CORES['destaque']],
        labels={'x': 'Safra', 'y': 'Variável', 'color': 'Ausência'}, zmin=0, zmax=max(0.01, missing_por_safra.max().max()),
    )
    aplicar_layout_executivo(fig_missing, 'Percentual de ausência por variável e safra',
                             titulo_eixo_x='Safra', titulo_eixo_y='Variável', altura=max(450, 45 * len(variaveis_com_missing)))
    fig_missing.update_coloraxes(colorbar_tickformat='.1%')
    fig_missing.show()
    salvar_grafico(fig_missing, '00_02_missing_por_safra', PASTA_FIGURAS)
display(Markdown(
    f"**Análise/Interpretação:** **{len(variaveis_com_missing)} variáveis** possuem ausência. "
    f"O maior percentual é **{tabela_missing['percentual_ausentes'].max():.2%}**. "
    "O mapa temporal permite distinguir ausência estruturalmente estável de mudanças concentradas em certas safras; "
    "nenhum valor foi imputado."
))

,variavel,quantidade_ausentes,percentual_ausentes,quantidade_valores_unicos
15,cat_var13,82692,41.34%,15
6,var4,1607,0.80%,37882
4,cat_var2,1323,0.66%,5
10,var8,546,0.27%,43219
11,var9,443,0.22%,13
13,var11,4,0.00%,50787
17,cat_var15,3,0.00%,13
2,Ever30Mob6,0,0.00%,2
12,cat_var10,0,0.00%,10
8,cat_var6,0,0.00%,10


C:\GitHub\datascience\projetos\credit-card-behavior-score\src\behavior_score\visualization.py:187: RuntimeWarning: Não foi possível exportar o gráfico para PNG. O arquivo HTML foi preservado. Detalhe: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

  warnings.warn(


**Análise/Interpretação:** **7 variáveis** possuem ausência. O maior percentual é **41.34%**. O mapa temporal permite distinguir ausência estruturalmente estável de mudanças concentradas em certas safras; nenhum valor foi imputado.

## 6. Como se comportam as variáveis numéricas e os códigos especiais?

Os valores `99997`, `99998` e `99999` são mantidos intactos e analisados como códigos de significado desconhecido.

In [7]:
numericas_presentes = [v for v in VARIAVEIS_NUMERICAS if v in base.columns]
quantis = base[numericas_presentes].quantile([.01, .05, .25, .50, .75, .95, .99]).T
quantis.columns = ['p01', 'p05', 'p25', 'mediana', 'p75', 'p95', 'p99']
tabela_numericas = pd.DataFrame({
    'quantidade': base[numericas_presentes].count(),
    'percentual_missing': base[numericas_presentes].isna().mean(),
    'valores_unicos': base[numericas_presentes].nunique(),
    'media': base[numericas_presentes].mean(),
    'desvio_padrao': base[numericas_presentes].std(),
    'minimo': base[numericas_presentes].min(),
    'maximo': base[numericas_presentes].max(),
}).join(quantis)[['quantidade', 'percentual_missing', 'valores_unicos', 'media', 'desvio_padrao',
                   'minimo', 'p01', 'p05', 'p25', 'mediana', 'p75', 'p95', 'p99', 'maximo']]
display(tabela_numericas.style.format({'percentual_missing': '{:.2%}'}))
valores_especiais = [99997, 99998, 99999]
registros_especiais = []
registros_especiais_safra = []
for variavel in numericas_presentes:
    for valor in valores_especiais:
        mascara = base[variavel].eq(valor)
        if mascara.any():
            taxa_com = base.loc[mascara, ALVO].mean()
            taxa_sem = base.loc[~mascara, ALVO].mean()
            registros_especiais.append({
                'variavel': variavel, 'valor_especial': valor, 'quantidade': int(mascara.sum()),
                'percentual': mascara.mean(), 'quantidade_safras': int(safra_data[mascara].nunique()),
                'taxa_evento_com_valor': taxa_com, 'taxa_evento_sem_valor': taxa_sem,
                'diferenca_taxa_evento': taxa_com - taxa_sem,
            })
            por_safra = (pd.DataFrame({'safra': safra_data, 'presente': mascara, ALVO: base[ALVO]})
                         .groupby('safra').agg(quantidade=('presente', 'sum'), percentual=('presente', 'mean'),
                                               taxa_evento=(ALVO, lambda s: s[mascara.loc[s.index]].mean())))
            por_safra = por_safra.reset_index().assign(variavel=variavel, valor_especial=valor)
            registros_especiais_safra.append(por_safra)
tabela_especiais = pd.DataFrame(registros_especiais)
tabela_especiais_safra = pd.concat(registros_especiais_safra, ignore_index=True) if registros_especiais_safra else pd.DataFrame()
display(tabela_especiais.style.format({c: '{:.2%}' for c in ['percentual', 'taxa_evento_com_valor',
                                                               'taxa_evento_sem_valor', 'diferenca_taxa_evento']}))
display(Markdown(
    f"**Análise/Interpretação:** **{tabela_especiais['variavel'].nunique() if not tabela_especiais.empty else 0} variáveis numéricas** "
    "contêm ao menos um código especial investigado. Diferenças de taxa são associações descritivas, não prova de leakage "
    "nem justificativa para converter esses códigos em ausência. O tratamento exige validação humana posterior."
))

,quantidade,percentual_missing,valores_unicos,media,desvio_padrao,minimo,p01,p05,p25,mediana,p75,p95,p99,maximo
var1,200043,0.00%,83948,438.134308,393.888086,0.000000,0.000000,0.000000,159.600000,340.400000,606.810000,1186.618000,1789.651600,8900.060000
var3,200043,0.00%,96508,510.307926,432.332837,5.000000,16.804200,58.880000,202.460000,402.600000,695.315000,1323.902000,1988.430800,15207.890000
var4,198436,0.80%,37882,0.914584,0.237842,0.000005,0.176848,0.362128,1.000000,1.000000,1.000000,1.000000,1.000102,34.336752
var5,200043,0.00%,47,0.876854,0.169364,0.000000,0.250000,0.500000,0.818182,0.916667,1.000000,1.000000,1.000000,1.000000
var7,200043,0.00%,99328,627.226348,750.605536,0.000000,0.000000,0.000000,123.960000,396.920000,871.920000,2002.938000,3425.032800,15948.750000
var8,199497,0.27%,43219,0.876121,0.248445,0.000005,0.154432,0.239514,0.978846,1.000000,1.000000,1.000000,1.000000,4.034723
var9,199600,0.22%,13,0.872329,0.204278,0.000000,0.166667,0.500000,0.833333,1.000000,1.000000,1.000000,1.000000,1.000000
var11,200039,0.00%,50787,0.824034,0.283376,0.000005,0.143061,0.216034,0.664878,1.000000,1.000000,1.000000,1.000000,2.000000
var12,200043,0.00%,10,93317.056774,24969.638560,0.333333,0.500000,1.000000,99998.000000,99999.000000,99999.000000,99999.000000,99999.000000,99999.000000
var14,200043,0.00%,13,0.806046,0.213772,0.000000,0.166667,0.333333,0.666667,0.916667,1.000000,1.000000,1.000000,1.000000


,variavel,valor_especial,quantidade,percentual,quantidade_safras,taxa_evento_com_valor,taxa_evento_sem_valor,diferenca_taxa_evento
0,var12,99997,22724,11.36%,13,24.66%,11.06%,13.60%
1,var12,99998,58776,29.38%,13,9.27%,14.00%,-4.73%
2,var12,99999,105177,52.58%,13,10.11%,15.38%,-5.27%


**Análise/Interpretação:** **1 variáveis numéricas** contêm ao menos um código especial investigado. Diferenças de taxa são associações descritivas, não prova de leakage nem justificativa para converter esses códigos em ausência. O tratamento exige validação humana posterior.

## 7. Qual é o diagnóstico das variáveis categóricas?

Categorias numericamente codificadas não recebem interpretação de negócio. Para esta triagem, categoria rara é apenas um sinal operacional com participação inferior a 1%; nenhuma categoria é agrupada ou removida.

In [8]:
categoricas_presentes = [v for v in VARIAVEIS_CATEGORICAS if v in base.columns]
diagnostico_categoricas = []
taxas_categorias = []
presenca_categorias_safra = []
for variavel in categoricas_presentes:
    frequencias = base[variavel].value_counts(dropna=False)
    participacoes = frequencias / len(base)
    diagnostico_categoricas.append({
        'variavel': variavel, 'cardinalidade_sem_missing': base[variavel].nunique(dropna=True),
        'quantidade_missing': int(base[variavel].isna().sum()), 'percentual_missing': base[variavel].isna().mean(),
        'categoria_dominante': frequencias.index[0], 'participacao_dominante': participacoes.iloc[0],
        'quantidade_categorias_raras_menor_1pct': int((participacoes < .01).sum()),
        'categorias_mais_frequentes': '; '.join(f'{str(k)} ({v:.1%})' for k, v in participacoes.head(5).items()),
    })
    taxas = (base.groupby(variavel, dropna=False)[ALVO].agg(quantidade='size', taxa_evento='mean')
             .reset_index().assign(variavel=variavel))
    taxas_categorias.append(taxas)
    presenca = (base.assign(safra=safra_data).groupby(['safra', variavel], dropna=False).size()
                .rename('quantidade').reset_index().assign(variavel_nome=variavel))
    presenca_categorias_safra.append(presenca)
tabela_categoricas = pd.DataFrame(diagnostico_categoricas)
tabela_taxas_categorias = pd.concat(taxas_categorias, ignore_index=True)
tabela_presenca_categorias_safra = pd.concat(presenca_categorias_safra, ignore_index=True)
display(tabela_categoricas.style.format({'percentual_missing': '{:.2%}', 'participacao_dominante': '{:.2%}'}))
display(tabela_taxas_categorias.sort_values(['variavel', 'quantidade'], ascending=[True, False]).head(50)
        .style.format({'taxa_evento': '{:.2%}'}))
display(Markdown(
    "**Análise/Interpretação:** A tabela consolida cardinalidade, ausência, concentração e categorias raras. "
    "As tabelas detalhadas preservam taxa do evento e presença temporal por código. Categorias raras ou instáveis são "
    "candidatas a investigação futura, não a exclusão automática."
))

,variavel,cardinalidade_sem_missing,quantidade_missing,percentual_missing,categoria_dominante,participacao_dominante,quantidade_categorias_raras_menor_1pct,categorias_mais_frequentes
0,cat_var2,5,1323,0.66%,1.000000,71.59%,2,1.0 (71.6%); 0.666666666666666 (16.7%); 0.333333333333333 (6.8%); 0.0 (3.2%); 0.5 (1.0%)
1,cat_var6,10,0,0.00%,12.000000,58.19%,0,12 (58.2%); 11 (9.3%); 3 (5.5%); 4 (4.8%); 5 (4.2%)
2,cat_var10,10,0,0.00%,0.000000,81.09%,0,0 (81.1%); 7 (3.2%); 6 (2.8%); 5 (2.4%); 8 (2.3%)
3,cat_var13,15,82692,41.34%,nan,41.34%,7,nan (41.3%); 2.0 (12.6%); 3.0 (10.1%); 1.0 (8.5%); 5.0 (7.8%)
4,cat_var15,13,3,0.00%,0.000000,29.02%,1,0.0 (29.0%); 1.0 (10.0%); 2.0 (8.9%); 3.0 (8.0%); 4.0 (6.6%)


,cat_var2,quantidade,taxa_evento,variavel,cat_var6,cat_var10,cat_var13,cat_var15
16,nan,162222,9.89%,cat_var10,nan,0.000000,nan,nan
23,nan,6481,26.43%,cat_var10,nan,7.000000,nan,nan
22,nan,5651,25.69%,cat_var10,nan,6.000000,nan,nan
21,nan,4731,22.93%,cat_var10,nan,5.000000,nan,nan
24,nan,4577,26.33%,cat_var10,nan,8.000000,nan,nan
20,nan,4205,22.31%,cat_var10,nan,4.000000,nan,nan
19,nan,3644,21.30%,cat_var10,nan,3.000000,nan,nan
18,nan,3146,22.19%,cat_var10,nan,2.000000,nan,nan
25,nan,2787,26.16%,cat_var10,nan,9.000000,nan,nan
17,nan,2599,22.35%,cat_var10,nan,1.000000,nan,nan


**Análise/Interpretação:** A tabela consolida cardinalidade, ausência, concentração e categorias raras. As tabelas detalhadas preservam taxa do evento e presença temporal por código. Categorias raras ou instáveis são candidatas a investigação futura, não a exclusão automática.

## 8. Existem constantes, baixa variabilidade ou sinais preliminares de leakage?

**Comentário Técnico:** O limiar de 99% para quase constante é uma regra diagnóstica provisória. A triagem de leakage usa associação univariada e taxas extremas somente para gerar alertas; forte predição não implica vazamento.

In [9]:
variaveis_auditadas = [v for v in VARIAVEIS_MODELO if v in base.columns]
tabela_variabilidade = []
for variavel in variaveis_auditadas:
    frequencias = base[variavel].value_counts(dropna=False, normalize=True)
    maior_participacao = float(frequencias.iloc[0]) if len(frequencias) else np.nan
    tabela_variabilidade.append({
        'variavel': variavel, 'valores_unicos_incluindo_missing': int(base[variavel].nunique(dropna=False)),
        'maior_participacao': maior_participacao, 'constante': base[variavel].nunique(dropna=False) <= 1,
        'quase_constante_99pct': maior_participacao >= .99 and base[variavel].nunique(dropna=False) > 1,
    })
tabela_variabilidade = pd.DataFrame(tabela_variabilidade).sort_values('maior_participacao', ascending=False)
alertas_leakage = []
for variavel in numericas_presentes:
    pares = base[[variavel, ALVO]].dropna()
    correlacao = pares[variavel].corr(pares[ALVO]) if pares[variavel].nunique() > 1 else np.nan
    if pd.notna(correlacao) and abs(correlacao) >= .80:
        alertas_leakage.append({'variavel': variavel, 'tipo_alerta': 'correlacao_linear_absoluta_ge_0_80',
                                 'evidencia': correlacao, 'observacao': 'triagem; investigar temporalidade e definição'})
for _, linha in tabela_taxas_categorias.iterrows():
    if linha['quantidade'] >= 100 and (linha['taxa_evento'] <= .005 or linha['taxa_evento'] >= .995):
        alertas_leakage.append({'variavel': linha['variavel'], 'tipo_alerta': 'categoria_com_taxa_extrema',
                                 'evidencia': linha['taxa_evento'], 'observacao': f"categoria={linha.iloc[0]!r}; n={linha['quantidade']}"})
if not tabela_especiais.empty:
    for _, linha in tabela_especiais.iterrows():
        if linha['quantidade'] >= 100 and abs(linha['diferenca_taxa_evento']) >= .20:
            alertas_leakage.append({'variavel': linha['variavel'], 'tipo_alerta': 'valor_especial_associacao_forte',
                                     'evidencia': linha['diferenca_taxa_evento'],
                                     'observacao': f"valor={int(linha['valor_especial'])}; associação descritiva"})
tabela_alertas_leakage = pd.DataFrame(alertas_leakage, columns=['variavel', 'tipo_alerta', 'evidencia', 'observacao'])
display(tabela_variabilidade.style.format({'maior_participacao': '{:.2%}'}))
display(tabela_alertas_leakage if not tabela_alertas_leakage.empty else pd.DataFrame({'resultado': ['nenhum alerta pelos critérios explícitos']}))
display(Markdown(
    f"**Análise/Interpretação:** Foram identificadas **{tabela_variabilidade['constante'].sum()} constantes**, "
    f"**{tabela_variabilidade['quase_constante_99pct'].sum()} quase constantes** pelo limiar provisório e "
    f"**{len(tabela_alertas_leakage)} alertas preliminares de leakage**. Nenhum resultado implica remoção. "
    "A disponibilidade temporal e a origem das variáveis precisam de validação humana para distinguir poder preditivo legítimo de vazamento."
))

,variavel,valores_unicos_incluindo_missing,maior_participacao,constante,quase_constante_99pct
12,cat_var10,10,81.09%,False,False
2,var4,37883,76.98%,False,False
10,cat_var2,6,71.59%,False,False
5,var8,43220,70.39%,False,False
6,var9,14,61.22%,False,False
7,var11,50788,60.30%,False,False
11,cat_var6,10,58.19%,False,False
8,var12,10,52.58%,False,False
3,var5,47,47.00%,False,False
13,cat_var13,16,41.34%,False,False


,resultado
0,nenhum alerta pelos critérios explícitos


**Análise/Interpretação:** Foram identificadas **0 constantes**, **0 quase constantes** pelo limiar provisório e **0 alertas preliminares de leakage**. Nenhum resultado implica remoção. A disponibilidade temporal e a origem das variáveis precisam de validação humana para distinguir poder preditivo legítimo de vazamento.

## 9. Resumo executivo e artefatos

As tabelas exportadas são derivadas; o Excel original permanece somente leitura.

In [10]:
resumo_executivo = pd.DataFrame({
    'metrica': ['quantidade_linhas', 'quantidade_colunas', 'numero_safras', 'primeira_safra', 'ultima_safra',
                'taxa_global_evento', 'ids_unicos', 'ids_repetidos', 'linhas_duplicadas',
                'variaveis_com_missing', 'maior_percentual_missing', 'variaveis_com_valores_especiais',
                'variaveis_numericas', 'variaveis_categoricas', 'variaveis_constantes',
                'variaveis_quase_constantes_99pct', 'alertas_preliminares_leakage'],
    'valor': [len(base), base.shape[1], len(tabela_safras), tabela_safras['safra'].min().strftime('%Y-%m'),
              tabela_safras['safra'].max().strftime('%Y-%m'), base[ALVO].mean(), ids_unicos, ids_repetidos,
              linhas_duplicadas, len(variaveis_com_missing), tabela_missing['percentual_ausentes'].max(),
              tabela_especiais['variavel'].nunique() if not tabela_especiais.empty else 0, len(numericas_presentes),
              len(categoricas_presentes), int(tabela_variabilidade['constante'].sum()),
              int(tabela_variabilidade['quase_constante_99pct'].sum()), len(tabela_alertas_leakage)],
})
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
tabelas_exportar = {
    '00_fonte.csv': tabela_fonte, '00_estrutura.csv': tabela_estrutura, '00_identificadores.csv': tabela_identificadores,
    '00_safras.csv': tabela_safras, '00_alvo_por_safra.csv': tabela_evento_safra,
    '00_missing.csv': tabela_missing, '00_numericas.csv': tabela_numericas.reset_index(names='variavel'),
    '00_valores_especiais.csv': tabela_especiais, '00_valores_especiais_por_safra.csv': tabela_especiais_safra,
    '00_categoricas.csv': tabela_categoricas, '00_taxa_evento_categorias.csv': tabela_taxas_categorias,
    '00_presenca_categorias_por_safra.csv': tabela_presenca_categorias_safra,
    '00_variabilidade.csv': tabela_variabilidade, '00_alertas_leakage.csv': tabela_alertas_leakage,
    '00_resumo_executivo.csv': resumo_executivo,
}
for nome, tabela in tabelas_exportar.items():
    tabela.to_csv(PASTA_TABELAS / nome, index=False, encoding='utf-8-sig')
display(resumo_executivo)
display(Markdown(
    "**Análise/Interpretação:** A auditoria fornece evidências para a próxima revisão metodológica, mas não decide "
    "tratamentos, exclusões ou janelas temporais. Devem ser validados por uma pessoa: a divergência de schema, a "
    "granularidade efetiva, o significado dos códigos especiais, os candidatos de baixa variabilidade e cada alerta de leakage."
))
assert base.equals(base_original), 'A base em memória foi alterada durante a auditoria.'
print(f'{len(tabelas_exportar)} tabelas derivadas exportadas para {PASTA_TABELAS.relative_to(RAIZ)}.')

,metrica,valor
0,quantidade_linhas,200043
1,quantidade_colunas,18
2,numero_safras,13
3,primeira_safra,2019-01
4,ultima_safra,2020-01
5,taxa_global_evento,0.126063
6,ids_unicos,200043
7,ids_repetidos,0
8,linhas_duplicadas,0
9,variaveis_com_missing,7


**Análise/Interpretação:** A auditoria fornece evidências para a próxima revisão metodológica, mas não decide tratamentos, exclusões ou janelas temporais. Devem ser validados por uma pessoa: a divergência de schema, a granularidade efetiva, o significado dos códigos especiais, os candidatos de baixa variabilidade e cada alerta de leakage.

15 tabelas derivadas exportadas para reports\tables.
